# **Primary Function**

**Ensure you have pressed "run all below" on the imports prior to using this function, as everything needs to be defined and loaded into memory to work!**

Type your command into the 'input' string and press run

In [ ]:
input_list = []
print("Dungeon Assistant ready! Type 'quit' to exit or 'reset' to restart the conversation.")
# Example inputs based on my current campaign:
# It is the start of session 2. The players are heading to a tavern on the outskirts of town. Darrow knows the tavern from his backstory, the tavern should be a cozy place for workers to rest, much like a pub, where he knows a handful of the people there. Can you give me a quick scene of the tavern, as well as a couple interesting characters to populate it?
# Sindri has started a fight in the tavern, and Warren is joining in the fight to protect him. I would just like a moderately difficult encounter inside and around the tavern, with a battlemap and statblocks. Please also roll initiative for the players and the enemies. I will add modifiers to the player's statistics, so just roll a D20 for each.
while True:
    user_input = input("\n-> ")
    print()
    if user_input.lower() == "quit":
        print("Ending session.")
        break
    elif user_input.lower() == "reset":
        input_list = []
        active_vs = None
        print("Session reset.\n")
        continue
    DungeonAssistant(user_input, "gpt-5.4")

Dungeon Assistant ready! Type 'quit' to exit or 'reset' to restart the conversation..

-> It is the start of session 2. The players are heading to a tavern on the outskirts of town. Darrow knows the tavern from his backstory, the tavern should be a cozy place for workers to rest, much like a pub, where he knows a handful of the people there. Can you give me a quick scene of the tavern, as well as a couple interesting characters to populate it?

 - Sending initial message to agent.
 - Accessing files
   - Campaign notes
 - Calling image generation API
  - Prompt: A cozy workers' tavern on the outskirts of a fantasy town at dusk, warm lantern light glowing through rain-streaked windows, sturdy timber beams, scarred wooden tables, a broad hearth, work boots by the door, tankards, regulars in plain work clothes relaxing after a long day, intimate pub atmosphere, grounded medieval fantasy, welcoming but lived-in
  - Saving image to disc
 - Accessing files
   - Notes
 - Taking notes
 - Acces

KeyboardInterrupt: Interrupted by user

# **Code & Implementation**


In [ ]:
!pip install openai
!pip install magic_hour
import openai
import magic_hour
import os
import requests
import time
import json
import base64
from PIL import Image
from zipfile import ZipFile
from google.colab import userdata

### Importing example data to local files
I have to download the file from my drive as colab local files are weird. Some of these are universal for everyone, some are specific to my campaign. In an actual agent you could access a local path for file detection and upload.

In [ ]:
# Link Drive files here, use google drive share ID to download,
# Sample notes used here
def prepare_files(campaign_dir: str = None):
    print(" - Preparing files and directories for agent.")
    # I dont know why google colab insists on having sample data
    if os.path.exists("/content/sample_data"):
        !rm -rf /content/sample_data

    # Download assistant files in google colab - These will always be used.
    if not os.path.exists("/content/DungeonAssistantResources"):
        !gdown 1fyRAigwjtxT5Eo0rkybZF693Y3ukAcxf -O DungeonAssitantResources.zip
        with ZipFile("/content/DungeonAssitantResources.zip", 'r') as zf:
            zf.extractall()
        !rm /content/DungeonAssitantResources.zip

    # Download campaign notes (if provided)
    if campaign_dir != None and not os.path.exists("/content/campaign"):
        !gdown $campaign_dir -O campaign.zip
        with ZipFile("/content/campaign.zip", 'r') as zf:
            zf.extractall()
        !rm /content/campaign.zip

    if not os.path.exists("/content/notes.txt"):
        f = open("/content/notes.txt", 'a')
        f.write("This is where you can take notes on the campaign. If it is currently empty there have been no notes taken previously.\n")
        f.close()

    if not os.path.exists("/content/images"):
        !mkdir /content/images

prepare_files("16csH140dZ03q_bCkyYzD8HqEqIwJdp-c")

 - Preparing files and directories for agent.


## Tools
First, we need to define the tools the responses api will have access too. This includes both their default tools like file search as well as custom functions
***
Tools:
- Retrieve files
- Retrieve object - API Call
  - Open5e has an API that I can call for retrieving any piece of information the model could need
- Play music through spotify account - API Call
  - Uses the Spotify API to adjust current track to fit the situation
- Image generation - API Call
  - Uses image generation API to create battlemaps and scenes for locations

- Encounter difficulty / reward rating
  - Takes monsters, fetches their CR rating and does a calculation for difficulty and reward value
- Dice Roller (code interpretation)
- Write notes
- Generate images

### Tool Formatting Functions

In [ ]:
# I am defining this function so I don't need to rewrite this syntax for new tools.
def create_tool(name: str, description: str, properties: dict = {}, required: list = []) -> dict:
    tool = {
        "type": "function",
        "name": name,
        "description": description,
    }

    if len(properties) > 0:
        tool["parameters"] = {
            "type": "object",
            "properties": properties,
        }
        if len(required) > 0:
            tool["parameters"]["required"] = required

    return tool

# Same here, this helps remove the need for excess formatting
def create_property(type: str = "string", array_type: str = "string", description: str = "", enum: list = []) -> dict:
    property = {}
    property["type"] = type
    if type == "array":
        property["items"] = {
            "type": array_type
        }

    if len(enum) > 0:
        property["enum"] = enum

    property["description"] = description
    return property

# The main tools list
tools = []

### Vector Store Functions

In [ ]:
# This function is used to create a vector store for things like file search.
def vector_store(client, name: str):
    vs = client.vector_stores.create(name=name, expires_after={
        "anchor": "last_active_at",
        "days": 1
    })
    return vs

# Adds multiple files to a vector store
def add_files(client, vector_store, files: list):
    for file in files:
        client.vector_stores.files.upload_and_poll(
            vector_store_id=vector_store.id,
            file=open(file, "rb"))

# Adds one file to a vector store
def add_file(client, vector_store, file: str):
    client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id,
        file=open(file, "rb")
    )

# Add new event to the input list
def add_event(input_list: list, role: str, content: str):
    input_list.append({"role": role, "content": content})

### File Retrieval
This is the primary RAG access. A user _can_ connect their campaign notes to the agent if they wish through a google file. Regardless of whether they do or not, the agent will have access to the core 2024 DnD rulebooks, alongside a statblock template. Access to more rulebooks could create for a better assistant, but acquiring the rulebooks in a PDF format is difficult for D&D.

In [ ]:
name = "get_file"
description = "Retrieves the three core rulebooks, a statblock template, or the DM's campaign files if provided."
files = ["Dungeon Master's Guide", "Player's Handbook", "Monster Manual", "Statblock Template", "Notes"]
if os.path.exists("/content/campaign"):
    print("Including campaign notes")
    files.append("Campaign Notes")
properties = {
    "file": create_property(type="string", description="File to retrieve", enum=files)
}
required = ["file"]

# Have it specifiy which files to save on tokens maybe?
tools.append(create_tool(name, description, properties, required))

vs = None

def get_file(client, filename: list):
    print(" - Accessing files")

    global vs
    if vs is None:
        vs = vector_store(client, "Requested Files")
        tools.append({
            "type": "file_search",
            "vector_store_ids": [vs.id]
        })

    match filename:
        case "Campaign Notes":
            print("   - Campaign notes")
            # Adds all files in "campaign" to the vector store
            temp_files = []
            for detected_file in os.scandir("/content/campaign"):
                temp_files.append(detected_file.path)
            add_files(client, vs, temp_files)

        case "Dungeon Master's Guide":
            print("   - DMG")
            add_file(client, vs, "/content/DungeonAssistantResources/dmg.pdf")

        case "Player's Handbook":
            print("   - PHB")
            add_file(client, vs, "/content/DungeonAssistantResources/phb.pdf")

        case "Monster Manual":
            print("   - MOM")
            add_file(client, vs, "/content/DungeonAssistantResources/mom.pdf")

        case "Statblock Template":
            print("   - Statblock")
            add_file(client, vs, "/content/DungeonAssistantResources/statblock_text.txt")

        case "Notes":
            print("   - Notes")
            add_file(client, vs, "/content/notes.txt")

    return vs

Including campaign notes


### Object Retrieval

This is the first API call of three. In this the agent will have access to detailed knowledge about pretty much any specific component of DnD. Where RAG and the rulebooks will give general advice and details, this provides specifics.

The API being called is Open5e, a community API that has access to a vast amount of knowledge

In [ ]:
name = "get_object"
description = "Retrieves information about most individual DnD objects."
open5e_options = ["spells","creatures", "backgrounds", "species", "environments", "feats", "conditions", "races", "classes", "magicitems"]
# I likely need to decide between specific inputs (type and name) or a query-based system.
properties = {
    "type": create_property(description= "Type of object to retrieve", enum=open5e_options),
    "name": create_property(description="Name of the object to retrieve"),
}
required = ["type", "name"]

tools.append(create_tool(name, description, properties, required))

def get_object(type: str, name: str):
    # Construct URL based on requested details
    print(" - Fetching information from Open5e")
    # The open5e api seems to be much better at searching when these extra search variables are specified
    url = f"https://api.open5e.com/v2/{type}/?key__in=&key__iexact=&key=&name__iexact=&name=&name__icontains={name.lower().replace(' ', '+')}&document__key__in=srd-2024&document__key__iexact=&document__key=&document__gamesystem__key__in=&document__gamesystem__key__iexact=&document__gamesystem__key="

    response = requests.get(url)

    if response.status_code == 200 and len(response.json()['results']) > 0:
        print(f"  - Response: {response.status_code}, Returning information to agent")
        return response.json()['results'][0]
    else:
        print(f"  - Response: {response.status_code}, Failed call, notfiying agent")
        return {"Status": "Error with query. Type and Name may not exist in database, or there may have been an issue with the API."}

### Music Integration

The second API call will use Spotify's web API to control music for a user if they provide their spotify information. In a proper application form of this agent you could use a spotify web SDK integration to make handling the spotify connection easier.

In [ ]:
# name = "play_music"
# description = "Changes the user's current song to the top song based on a Spotify search query."
# properties = {
#     "query": create_property(description="Query used to find a song of a specific theme, genre, artist or more.")
# }
# required = ["query"]

# tools.append(create_tool(name, description, properties, required))

# def play_music():
#     #TODO
#     #
#     return None

### Image Generation

The third and final API will use Magic Hour's image generator. I am using their image generator as it is reasonably popular, includes a DnD themed style, and includes ~100 free image generations.

Unfortunately using these free credits means there is a watermark on the images. For a proper application that would require some sort of income to run, this could be paid for.

For a return, it will save the image generated and feed it back to the AI using A vector store. I am not sure how much this will actually help the agent, but it might be able to generate multiple images until it is happy. This needs further testing.

In [ ]:
name = "generate_image"
description = "Generates an image based off of a prompt. The image will be shown to the user and stored locally."
properties = {
    "prompt": create_property(description="Prompt to generate image. Being more specific tends to create a better image."),
    "file_name": create_property(description="File name for the user to recognize the generated image")
}
required = ["prompt", "file_name"]

tools.append(create_tool(name, description, properties, required))

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def generate_image(prompt: str, file_name: int):
    print(" - Calling image generation API")
    print(f"  - Prompt: {prompt}")
    image_gen = magic_hour.Client(token=userdata.get('magic_hour_key'))

    MAX_RETRIES = 3
    RETRY_DELAY = 5

    # Loop retries call if error occurs
    for attempt in range(MAX_RETRIES):
        try:
            res = image_gen.v1.ai_image_generator.generate(
                image_count=1,
                style={
                    "prompt": prompt,
                    "tool": "dnd-ai-art-generator",
                },
                aspect_ratio="16:9",
                name=f"{file_name}",
                model="flux-schnell",
                resolution="auto",
                wait_for_completion=True,
                download_outputs=True,
                download_directory=f"."
            )

            if res == None:
                raise Exception("No response from image generation API")

            print("  - Saving image to disc")
            os.rename(res.downloaded_paths[0], f"/content/images/{file_name}.png")
            image = Image.open(f"/content/images/{file_name}.png")
            image.show()
            return # Exit function if call is sucessful
        except Exception as e:
            print(f"  - An error occurred on attempt {attempt + 1}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES - 1:
                print(f"  - Retrying in {RETRY_DELAY} seconds...")
                time.sleep(RETRY_DELAY)
                RETRY_DELAY = 5
            else:
                print("  - Max retries reached. Failing image generation.")
                raise # Re-raise the exception after last attempt

### Encounter calculator

In [ ]:
name = "encounter_calculator"
description = "Compares monster and player levels to determine difficulty. Location should also be considered, but can only be done case-by-case."
properties = {
    "xp": create_property(type="array", array_type="integer", description="Experience reward of each monster in the encounter"),
    "levels": create_property(type="array", array_type="integer", description="Levels of all the players in the encounter")
}
required = ["xp", "levels"]

tools.append(create_tool(name, description, properties, required))

def encounter_calculator(crs: list, levels: list):
    print(" - Calculating encounter difficulties.")
    player_count = len(levels)
    xppp = sum(crs) / float(player_count) # eXPerience Per Player
    challenge_sum = 0

    # Challenge rating is, unfortunately, not a function. Thus this dictionary
    difficulty_table = {
        1:  (50, 100, 150),     2:  (100, 150, 200),     3:  (150, 225, 400),     4:  (250, 375, 500),      5:  (500, 750, 1100),
        6:  (600, 1000, 1400),  7:  (750, 1300, 1700),   8:  (1000, 1700, 2100),  9:  (1300, 2000, 2600),   10: (1600, 2300, 3100),
        11: (1900, 2900, 4100), 12: (2200, 3700, 4700),  13: (2600, 4200, 5400),  14: (2900, 4900, 6200),   15: (3300, 5400, 7800),
        16: (3800, 6100, 9800), 17: (4500, 7200, 11700), 18: (5000, 8700, 14200), 19: (5500, 10700, 17200), 20: (6400, 13200, 22000)
        }

    for player in levels:
        easy, medium, hard = difficulty_table[player]
        if xppp <= easy:
            pass # Here for posterity, we will use 0 to represent easy
        elif xppp <= medium:
            challenge_sum += 1
        elif xppp <= hard:
            challenge_sum += 2
        else:
            challenge_sum += 3

    if challenge_sum <= 0:
        print("  - Calculated difficulty: Very Easy")
        return {"difficulty": f"Extremely Easy", "reward": "None", "difficulties": [easy, medium, hard]}
    elif challenge_sum <= player_count:
        print("  - Calculated difficulty: Easy")
        return {"difficulty": "Easy", "reward": "Low", "difficulties": [easy, medium, hard]}
    elif challenge_sum <= player_count * 2:
        print("  - Calculated difficulty: Moderate")
        return {"difficulty": "Medium", "reward": "Moderate", "difficulties": [easy, medium, hard]}
    elif challenge_sum <= player_count * 3:
        print("  - Calculated difficulty: Challenging")
        return {"difficulty": "Hard", "reward": "High", "difficulties": [easy, medium, hard]}
    else:
        print("  - Calculated difficulty: Impossible")
        return {"difficulty": "Very Hard. Encounter will almost certainly result in a total party kill and should be redesigned.", "reward": "Extreme", "difficulties": [easy, medium, hard]}

### Dice Roller

In [ ]:
name = "dice_roller"
description = "Creates a code interpetation contained that can be used to roll dice. If the user asks for dice to be rolled a code interpreter should be used alongside python code in order to ensure randomness in the roles"

tools.append(create_tool(name, description))

def dice_roller(client):
    print(" - Creating container for dice rolling")
    container = client.containers.create(name="dice-roller", memory_limit="1g")
    tools.append({
        "type": "code_interpreter",
        "container": container.id
    })
    return container

### Notes

In [ ]:
name = "notes"
description = "Takes notes as specified and attaches file to vector storage."
properties = {
    "notes": create_property(type="string", description="Notes to take")
}
required = ["notes"]

tools.append(create_tool(name, description, properties, required))

# Extremely simple note taking feature, this is simply for
def notes(note: str):
    print(" - Taking notes")
    with open("/content/notes.txt", 'a') as f:
        f.write(f"{note}\n")
    return


## Main function

In [ ]:
input_list = []

def DungeonAssistant(input: str, model):
    instructions = """You are the assistant of a Dungeon Master for a game of 2024 5e DnD. Respond concisely and to the point.

    You MUST follow these rules at all times:
    - ALWAYS use notes from past conversations and the current campaign to inform creative decisions. Nothing should exist in a vaccuum.
    - ALWAYS use get_file to retrieve the relevant rulebook before answering any rules question. Never rely on your own knowledge for rules — the 2024 rulebooks may differ from your training data.
    - ALWAYS use get_object when asked about a specific spell, creature, item, condition, or feat. Do not describe these from memory.
    - ALWAYS use encounter_calculator when designing or evaluating encounters. Never estimate difficulty manually.
    - ALWAYS generate an image when creating a new location, scene, or battlemap.
    - ALWAYS take notes after learning new information about the campaign, players, or story events.
    - ALWAYS roll dice using the dice_roller tool and a python script rather than simulating results yourself.
    """

    global input_list
    if len(input_list) == 0:
        add_event(input_list, "developer", instructions)

    client = openai.OpenAI(api_key=userdata.get('oai_key'), timeout=240.0)
    add_event(input_list, "user", input)
    thinking = True

    print(" - Sending initial message to agent.")

    MAX_RETRIES = 3
    RETRY_DELAY = 2

    try:
        while thinking:
            # Loop to retry call if an error occurs.
            response = None
            for attempt in range(MAX_RETRIES):
                try:
                    response = client.responses.create(
                        model=model,
                        input=input_list,
                        tools=tools
                    )
                    break # Success, exit retry loop
                except Exception as e:
                    print(f" - An unexpected error during OpenAI call on attempt {attempt + 1}/{MAX_RETRIES}: {e}")
                    if attempt < MAX_RETRIES - 1:
                        print(f" - Retrying in {RETRY_DELAY} seconds...")
                        time.sleep(RETRY_DELAY)
                        RETRY_DELAY += 2
                    else:
                        print(" - Max API retries reached. Failing OpenAI call.")
                        raise # Re-raise after max retries
            if response is None: # If all retries failed and no response, something went wrong
                raise Exception("Failed to get a response from OpenAI API after multiple retries.")

            input_list.extend(response.output)

            # Detect what tool is called, and call it
            function_called = False
            for item in response.output:
                if item.type == "function_call":
                    function_called = True
                    arguments = json.loads(item.arguments)

                    match item.name:
                        case "get_file":
                            vs = get_file(client, arguments["file"])

                            input_list.append({
                                "type": "function_call_output",
                                "call_id": item.call_id,
                                "output": f"Vectore store sucessfully created and tools list updated.",
                            })


                        case "get_object":
                            object_info = get_object(arguments["type"], arguments["name"])

                            input_list.append({
                                "type": "function_call_output",
                                "call_id": item.call_id,
                                "output": json.dumps(object_info),
                            })


                        case "play_music":
                            pass # Unimplemented
                            input_list.append({
                                "type": "function_call_output",
                                "call_id": item.call_id,
                                "output": "Music playback not yet implemented.",
                            })

                        case "generate_image":
                            # The generate_image function itself now handles retries
                            image = generate_image(arguments["prompt"], arguments["file_name"])

                            input_list.append({
                                "type": "function_call_output",
                                "call_id": item.call_id,
                                "output": "Image generated and shown to user."
                            })


                        case "encounter_calculator":
                            difficulty = encounter_calculator(arguments["xp"], arguments["levels"])

                            input_list.append({
                                "type": "function_call_output",
                                "call_id": item.call_id,
                                "output": json.dumps(difficulty),
                            })


                        case "dice_roller":
                            container = dice_roller(client)

                            input_list.append({
                                "type": "function_call_output",
                                "call_id": item.call_id,
                                "output": "Container sucessfully created."
                            })


                        case "notes":
                            notes(arguments["notes"])

                            get_file(client, "Notes")

                            input_list.append({
                                "type": "function_call_output",
                                "call_id": item.call_id,
                                "output": "Notes taken and vector sucessfully store created with notes inside."
                            })

                elif item.type == "message":
                    print("\n" + response.output_text)
            thinking = function_called
        # This is to mark the end of the while loop, which should run until the API stops calling functions
    except Exception as e:
        input_list = []
        print("\nEncountered fatal error during process, please try again. This is likely due to an API failure. Past conversation has been wiped to prevent further errors.")
        print(e)